# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add your Hugging Face READ token "
        "as a Colab Secret named HF_TOKEN."
    )

print("HF_TOKEN found.")

import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face.")

march_df = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        client_has_gsc,
        client_has_ga4,
        gsc_data_available,
        ga4_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        scroll_events,
        sessions_ai,
        sessions_paid,
        sessions_direct,
        sessions_social,
        sessions_organic,
        month
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("March 2026 shape:", march_df.shape)
print(
    "Date range:",
    march_df["report_date"].min(),
    "to",
    march_df["report_date"].max()
)

display(march_df.head())

HF_TOKEN found.
DuckDB connected to Hugging Face.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 shape: (9841378, 17)
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai,sessions_paid,sessions_direct,sessions_social,sessions_organic,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Paper Finding 1 — Content Lifecycle / Age

The paper reports that growing pages were younger on average than declining pages: about 185 days versus 228 days. The groups were formed from pages gaining versus losing traffic.

**My methodology question:** Where exactly does the growing/declining label come from, and are all features used in this comparison measured before the period used to define that label? I would want to confirm that the comparison does not use information from the outcome period itself.

This is a constructive question because the finding is useful as an observed portfolio pattern, but the label definition and feature timing determine how strongly it can be generalized.

### Paper Finding 2 — The Freshness Multiplier

The paper reports that pages refreshed in the 365+ age group had higher health and impressions than an older stale comparison group, including a reported 1.6x health lift and 52x impression lift. It also reports a 5.43:1 growth-to-decline ratio for the 31–90 day freshness window.

**My methodology question:** Were pages selected for refresh based on characteristics that also affect later performance? If so, how much of the observed difference could come from selection rather than the refresh itself? I would also check whether the validation window is strictly after the refresh and whether comparable untouched pages were used.

This does not challenge the observed difference; it identifies what additional design would be needed before interpreting the comparison as evidence that refreshing itself caused the improvement.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

paper_audit = pd.DataFrame({
    "finding": [
        "Content Lifecycle / Age",
        "Freshness Multiplier"
    ],
    "methodology_question": [
        "Where does the growing/declining label come from, and are features measured before the label window?",
        "Were refreshed pages selected using characteristics that could also affect later performance?"
    ]
})

display(paper_audit)

,finding,methodology_question
0,Content Lifecycle / Age,Where does the growing/declining label come fr...
1,Freshness Multiplier,Were refreshed pages selected using characteri...


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split

In Week 5, I used a Random Forest regression model to estimate observed `scroll_events` from search and organic-session signals.

For this audit, I compare:

1. **Before:** a random row-level 80/20 split.
2. **After:** an honest grouped split where entire clients are held out from training.

The grouped split is more appropriate for this dataset because multiple content rows belong to the same client. The model therefore needs to be evaluated on clients it did not see during training.

The metric used for the model comparison is Spearman rank correlation between the observed `scroll_events` and the model's predicted values. This is aligned with the ranking/decision-support use of the model rather than treating the model as a causal explanation.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

model_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "sessions_organic",
    "scroll_events"
]

model_df = march_df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id"
    ] + model_cols
].copy()

for col in model_cols:
    model_df[col] = pd.to_numeric(
        model_df[col],
        errors="coerce"
    )

model_df = model_df.dropna(
    subset=[
        "gsc_impressions",
        "gsc_clicks",
        "gsc_sum_position",
        "scroll_events"
    ]
).copy()

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "sessions_organic"
]

X = model_df[feature_cols].copy()
y = model_df["scroll_events"].copy()
groups = model_df["client_hash_id"].copy()

print("Model rows:", len(model_df))
print("Unique clients:", groups.nunique())
print("Features:", feature_cols)

Model rows: 6822637
Unique clients: 43
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'sessions_organic']


In [8]:
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

random_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

random_model.fit(
    X_train_random,
    y_train_random
)

random_pred = random_model.predict(X_test_random)

random_mae = mean_absolute_error(
    y_test_random,
    random_pred
)

random_spearman = spearmanr(
    y_test_random,
    random_pred
).statistic

print("RANDOM ROW SPLIT")
print("Test rows:", len(X_test_random))
print("MAE:", round(random_mae, 4))
print("Spearman:", round(random_spearman, 4))

RANDOM ROW SPLIT
Test rows: 1364528
MAE: 0.0511
Spearman: 0.2149


In [10]:
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

group_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

group_model.fit(
    X_train_group,
    y_train_group
)

group_pred = group_model.predict(X_test_group)

group_mae = mean_absolute_error(
    y_test_group,
    group_pred
)

group_spearman = spearmanr(
    y_test_group,
    group_pred
).statistic

print("GROUPED CLIENT SPLIT")
print("Train rows:", len(X_train_group))
print("Test rows:", len(X_test_group))
print("Train clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())
print("MAE:", round(group_mae, 4))
print("Spearman:", round(group_spearman, 4))

GROUPED CLIENT SPLIT
Train rows: 4879304
Test rows: 1943333
Train clients: 34
Test clients: 9
MAE: 0.0817
Spearman: 0.2759


In [18]:
validation_comparison = pd.DataFrame({
    "split": [
        "Random row split",
        "Grouped by client"
    ],
    "test_rows": [
        len(X_test_random),
        len(X_test_group)
    ],
    "test_clients": [
        groups[X_test_random.index].nunique(),
        groups_test.nunique()
    ],
    "MAE": [
        random_mae,
        group_mae
    ],
    "Spearman": [
        random_spearman,
        group_spearman
    ]
})

display(validation_comparison)

spearman_gap = random_spearman - group_spearman

print(
    "Spearman gap (random - grouped):",
    round(spearman_gap, 4)
)

,split,test_rows,test_clients,MAE,Spearman
0,Random row split,1364528,43,0.051081,0.214915
1,Grouped by client,1943333,9,0.081652,0.275906


Spearman gap (random - grouped): -0.061


### Before/after interpretation

The random row split and grouped client split measure different levels of generalization.

The grouped split is the more honest validation for this dataset because client identity repeats across many content rows. Therefore, the grouped result is the number I use when describing how the model performs on unseen clients.

Any gap between the random and grouped result is treated as a validation finding, not as evidence that the model is universally better or worse.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I checked the final Week-5 feature set against three leakage risks:

1. Label-derived features
2. Future or overlapping outcome windows
3. Existing decision/product flags

The final model features are:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_sum_position`
- `sessions_organic`

The target used in Week 5 is observed `scroll_events`.

I do not use IDs as model features, and I do not use trend/label fields or existing action/decision flags as model inputs.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

final_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "sessions_organic"
]

suspect_terms = [
    "trend",
    "target",
    "label",
    "future",
    "next",
    "outcome",
    "action",
    "score",
    "flag"
]

feature_audit = []

for col in final_features:
    lower_col = col.lower()

    suspicious_name = any(
        term in lower_col
        for term in suspect_terms
    )

    feature_audit.append({
        "feature": col,
        "numeric": pd.api.types.is_numeric_dtype(march_df[col]),
        "suspicious_name": suspicious_name,
        "missing_rate": round(
            march_df[col].isna().mean(),
            4
        )
    })

feature_audit_df = pd.DataFrame(feature_audit)

display(feature_audit_df)

,feature,numeric,suspicious_name,missing_rate
0,gsc_impressions,True,False,0.0000
1,gsc_clicks,True,False,0.0000
2,gsc_sum_position,True,False,0.0000
3,sessions_organic,True,False,0.3067


In [20]:
dangerous_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_hash_id",
    "content_hash_id"
]

dangerous_present = [
    col for col in dangerous_columns
    if col in march_df.columns
]

print("Potentially dangerous columns present in data:")
print(dangerous_present)

print("\nFinal model features:")
print(final_features)

print("\nNo dangerous columns are included in the final feature list:")
print(
    set(final_features).isdisjoint(
        set(dangerous_present)
    )
)

Potentially dangerous columns present in data:
['client_hash_id', 'content_hash_id']

Final model features:
['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'sessions_organic']

No dangerous columns are included in the final feature list:
True


In [21]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": group_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

display(importance_df)

,feature,importance
0,sessions_organic,0.814310
1,gsc_impressions,0.088140
2,gsc_sum_position,0.080506
3,gsc_clicks,0.017044


### Real failure examples

I inspect the largest absolute prediction errors from the grouped-client holdout. These are examples where the model's observed engagement estimate differs substantially from the measured `scroll_events`.

These examples are used for diagnosis rather than claiming a universal failure pattern.

In [22]:
failure_examples = X_test_group.copy()

failure_examples["client_hash_id"] = groups_test.values
failure_examples["actual_scroll_events"] = y_test_group.values
failure_examples["predicted_scroll_events"] = group_pred

failure_examples["absolute_error"] = (
    failure_examples["actual_scroll_events"]
    - failure_examples["predicted_scroll_events"]
).abs()

failure_examples["signed_error"] = (
    failure_examples["predicted_scroll_events"]
    - failure_examples["actual_scroll_events"]
)

failure_examples = failure_examples.sort_values(
    "absolute_error",
    ascending=False
)

display(
    failure_examples.head(10)
)

,gsc_impressions,gsc_clicks,gsc_sum_position,sessions_organic,client_hash_id,actual_scroll_events,predicted_scroll_events,absolute_error,signed_error
5044300,2225,2,67267,2,client_23a62021009f63c4,93,1.565271,91.434729,-91.434729
5043736,5437,3,184683,3,client_23a62021009f63c4,87,0.715161,86.284839,-86.284839
5289454,5558,7,182829,8,client_23a62021009f63c4,66,1.638612,64.361388,-64.361388
5290018,2837,1,79926,0,client_23a62021009f63c4,61,0.113070,60.88693,-60.88693
4500131,514,1,7601,1,client_23a62021009f63c4,60,0.123866,59.876134,-59.876134
3616314,551,1,21100,0,client_23a62021009f63c4,59,0.130248,58.869752,-58.869752
5289618,781,3,20727,0,client_23a62021009f63c4,55,0.154459,54.845541,-54.845541
5043812,27,0,1084,0,client_23a62021009f63c4,54,0.049891,53.950109,-53.950109
4811302,2146,2,56401,0,client_23a62021009f63c4,53,0.103567,52.896433,-52.896433
5291540,144,4,1448,0,client_23a62021009f63c4,52,0.117241,51.882759,-51.882759


In [23]:
failure_summary = failure_examples.head(10)[
    [
        "client_hash_id",
        "actual_scroll_events",
        "predicted_scroll_events",
        "absolute_error",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_sum_position",
        "sessions_organic"
    ]
].copy()

display(failure_summary)

,client_hash_id,actual_scroll_events,predicted_scroll_events,absolute_error,gsc_impressions,gsc_clicks,gsc_sum_position,sessions_organic
5044300,client_23a62021009f63c4,93,1.565271,91.434729,2225,2,67267,2
5043736,client_23a62021009f63c4,87,0.715161,86.284839,5437,3,184683,3
5289454,client_23a62021009f63c4,66,1.638612,64.361388,5558,7,182829,8
5290018,client_23a62021009f63c4,61,0.113070,60.88693,2837,1,79926,0
4500131,client_23a62021009f63c4,60,0.123866,59.876134,514,1,7601,1
3616314,client_23a62021009f63c4,59,0.130248,58.869752,551,1,21100,0
5289618,client_23a62021009f63c4,55,0.154459,54.845541,781,3,20727,0
5043812,client_23a62021009f63c4,54,0.049891,53.950109,27,0,1084,0
4811302,client_23a62021009f63c4,53,0.103567,52.896433,2146,2,56401,0
5291540,client_23a62021009f63c4,52,0.117241,51.882759,144,4,1448,0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### My original claim

"My Random Forest predicts which content will get better engagement."

### Safer rewrite

"On the March 2026 warehouse slice, the Random Forest produced an out-of-sample ranking of observed `scroll_events`. Under grouped-by-client validation, the measured Spearman correlation was [INSERT GROUPED SPEARMAN VALUE]. This makes the model a directional decision-support signal for prioritizing pages for review, not evidence that the model will cause engagement to increase."

### Why I changed it

The original wording was too broad because the model was evaluated on one development window and one observed engagement measure. The safer wording states the dataset, metric, validation design, and decision-support use without claiming causality or universal predictive performance.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.